# 08. Uji kestabilan dan sensitivitas

Semua eksperimen di notebook ini memakai data 2021-2025. Tahun 2026 tidak digunakan untuk memilih parameter.

[Panduan](../00_panduan.ipynb) | [Sebelumnya](../07_temporal_analysis/07_temporal_analysis.ipynb) | [Berikutnya](../09_testing_2026/09_testing_2026.ipynb)

### Daftar bagian

- [1. Inisialisasi dan dependensi](#s1)
- [2. ARI parameter, grid, dan ambang magnitudo](#s2)
- [3. Jalankan eksperimen sensitivitas](#s3)
- [4. Sensitivitas kebijakan pemilihan DBSCAN](#s4)
- [5. Sensitivitas jumlah event per grid-triwulan](#s5)

<a id="s1"></a>
## 1. Inisialisasi dan dependensi

Jalankan dari atas ke bawah dengan kernel Python 3.12.11 (Seismic Research). Konfigurasi tersentral di config.json.

In [1]:
from pathlib import Path
import os
import sys

os.environ.setdefault("LOKY_MAX_CPU_COUNT", "2")
os.environ.setdefault("OMP_NUM_THREADS", "2")
candidates = []
for parent in [Path.cwd(), *Path.cwd().parents]:
    candidates.extend([parent, parent / "seismic_research_2021_2025"])
ROOT = next((p for p in candidates if (p / "config.json").is_file() and (p / "src/common.py").is_file()), None)
if ROOT is None:
    raise FileNotFoundError("Buka notebook dari folder proyek seismic_research_2021_2025.")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from IPython.display import display, Image, Markdown, FileLink
from src.common import read_config
cfg = read_config(ROOT)
print("Interpreter:", sys.executable)
print("Python:", sys.version.split()[0])
print("Proyek:", ROOT)
print("Training:", cfg["train_start"], "sampai sebelum", cfg["train_end_exclusive"])
print("Testing sampai sebelum (UTC):", cfg["test_end_exclusive"])

from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.metrics import adjusted_rand_score
from sklearn.neighbors import NearestNeighbors
from sklearn.cluster import KMeans, DBSCAN
from sklearn.preprocessing import StandardScaler
from threadpoolctl import threadpool_limits
from src.common import ROOT, load_model, load_prepared, write_json
from src.modeling import cluster_metrics
from src.preprocessing import grid_panel

Interpreter: C:\Users\yudhi\.local\bin\python3.12.exe
Python: 3.12.11
Proyek: C:\Users\yudhi\Documents\machine-learning\seismic_research_2021_2025
Training: 2021-01-01 sampai sebelum 2026-01-01
Testing sampai sebelum (UTC): 2026-10-26


<a id="s2"></a>
## 2. ARI parameter, grid, dan ambang magnitudo

Uji seed dan k K-Means; eps serta min_samples DBSCAN; grid 25/50/100 km; M minimum 2.5/3/3.5/4. Untuk grid berbeda, ARI dibandingkan pada **event ID historis yang sama** setelah setiap event diberi label grid-triwulannya. Hasil bersifat event-weighted, mengecualikan grid kosong dan melaporkan overlap. Bukan ARI langsung antara baris grid yang berlainan. Parameter terpilih dipertahankan, scaler di-fit ulang pada training alternatif.

In [2]:
def stability(root=ROOT):
    root = Path(root)
    cfg, _, train, X, _, _ = load_prepared(root)
    km, _ = load_model(root, "kmeans")
    db, _ = load_model(root, "dbscan")
    rows = []
    with threadpool_limits(limits=2):
        for k in sorted(set([max(2, km.n_clusters-1), km.n_clusters, km.n_clusters+1])):
            labels = KMeans(n_clusters=k, n_init=20, random_state=cfg["random_state"]).fit_predict(X)
            rows.append({"method": "kmeans", "perturbation": "n_clusters", "value": k, "n_same_rows":len(X),
                         "ari_all":adjusted_rand_score(km.labels_, labels), "ari_common_assigned":adjusted_rand_score(km.labels_, labels), "common_assigned_fraction":1.0})
        for seed in cfg["stability_seeds"]:
            labels = KMeans(n_clusters=km.n_clusters, n_init=20, random_state=seed).fit_predict(X)
            rows.append({"method": "kmeans", "perturbation": "random_seed", "value": seed, "n_same_rows": len(X),
                         "ari_all": adjusted_rand_score(km.labels_, labels), "ari_common_assigned": adjusted_rand_score(km.labels_, labels), "common_assigned_fraction": 1.0})
        for factor in [.8, 1.0, 1.2]:
            for min_samples in sorted(set([max(2, db.min_samples//2), db.min_samples, db.min_samples*2])):
                labels = DBSCAN(eps=db.eps*factor, min_samples=min_samples, n_jobs=1).fit_predict(X)
                common = (db.labels_ >= 0) & (labels >= 0)
                rows.append({"method": "dbscan", "perturbation": "eps_and_min_samples", "value": f"eps_x{factor};min_samples={min_samples}",
                             "n_same_rows": len(X), "ari_all": adjusted_rand_score(db.labels_, labels),
                             "ari_common_assigned": adjusted_rand_score(db.labels_[common], labels[common]) if common.sum()>1 else np.nan,
                             "common_assigned_fraction": common.mean()})
    params = pd.DataFrame(rows)
    params.to_csv(root / "reports/stability/parameter_ari.csv", index=False)
    support_rows=[]
    with threadpool_limits(limits=2):
        for minimum_count in [1,3,5,10]:
            mask=train["earthquake_count"].ge(minimum_count).to_numpy()
            subset=train.loc[mask]
            if len(subset)<=km.n_clusters:
                continue
            xx=StandardScaler().fit_transform(subset[cfg["features"]])
            for method,reference in [("kmeans",km.labels_),("dbscan",db.labels_)]:
                model=KMeans(n_clusters=km.n_clusters,n_init=20,random_state=cfg["random_state"]) if method=="kmeans" else DBSCAN(eps=db.eps,min_samples=db.min_samples,n_jobs=1)
                alternate=model.fit_predict(xx)
                common=(reference[mask]>=0)&(alternate>=0)
                support_rows.append({"method":method,"min_events_per_quarter":minimum_count,"n_same_grid_quarters":int(mask.sum()),
                                     "fraction_reference_rows":mask.mean(),"n_clusters":len(set(alternate)-{-1}),"noise_fraction":float((alternate<0).mean()),
                                     "ari_same_grid_quarters":adjusted_rand_score(reference[mask],alternate),
                                     "ari_common_assigned":adjusted_rand_score(reference[mask][common],alternate[common]) if common.sum()>1 else np.nan,
                                     "common_assigned_fraction":common.mean()})
    pd.DataFrame(support_rows).to_csv(root / "reports/stability/minimum_event_support.csv",index=False)
    search = pd.read_csv(root / "models/dbscan/parameter_search.csv")
    policies = []
    for max_clusters in [10, 20, 50, 300]:
        for min_cluster_size in [5, 20, 50]:
            valid = search.loc[search["n_clusters"].between(2,max_clusters)
                               & search["smallest_cluster_size"].ge(min_cluster_size)
                               & search["noise_fraction"].le(cfg["dbscan_max_noise_fraction"])
                               & search["largest_cluster_fraction_assigned"].le(cfg["dbscan_max_largest_cluster_fraction"])
                               & search["silhouette"].notna()]
            if valid.empty:
                policies.append({"max_clusters":max_clusters,"min_cluster_size":min_cluster_size,"status":"no_candidate"})
            else:
                selected=valid.sort_values(["silhouette","noise_fraction"],ascending=[False,True]).iloc[0]
                policies.append({"max_clusters":max_clusters,"min_cluster_size":min_cluster_size,"status":"candidate", **selected[["eps","min_samples","n_clusters","noise_fraction","silhouette"]].to_dict()})
    pd.DataFrame(policies).to_csv(root / "reports/stability/dbscan_selection_policy.csv", index=False)
    events = pd.read_csv(root / "data/processed/events_clean.csv")
    events["time"] = pd.to_datetime(events["time"], utc=True, format="mixed")
    reference_events = pd.read_csv(root / "data/processed/events_with_grid.csv")
    reference_events = reference_events.loc[reference_events["year"].between(2021,2025)]
    reference_events["row_id"] = reference_events["grid_id"] + "|" + reference_events["period"]
    base = reference_events[["event_key", "row_id"]].merge(train[["row_id"]].assign(kmeans=km.labels_, dbscan=db.labels_), on="row_id", how="inner")
    tests = [("grid_km", size) for size in cfg["grid_sensitivity_km"]]
    tests += [("min_magnitude", value) for value in cfg["magnitude_sensitivity"]]
    resolution_rows = []
    with threadpool_limits(limits=2):
        for kind, value in tests:
            alternate_cfg = {**cfg, kind: value}
            alt_panel, alt_events = grid_panel(events, alternate_cfg)
            alt_train = alt_panel.loc[alt_panel["split"].eq("train") & alt_panel["state"].eq("ELIGIBLE")].sort_values("row_id").reset_index(drop=True)
            if len(alt_train) <= km.n_clusters:
                continue
            xx = StandardScaler().fit_transform(alt_train[cfg["features"]])
            alt_events = alt_events.loc[alt_events["year"].between(2021,2025)].copy()
            alt_events["row_id"] = alt_events["grid_id"] + "|" + alt_events["period"]
            for method, ref_model in [("kmeans",km),("dbscan",db)]:
                model = KMeans(n_clusters=km.n_clusters, n_init=20, random_state=cfg["random_state"]) if method=="kmeans" else DBSCAN(eps=db.eps, min_samples=db.min_samples, n_jobs=1)
                labels = model.fit_predict(xx)
                membership = alt_events[["event_key", "row_id"]].merge(alt_train[["row_id"]].assign(alternate_label=labels), on="row_id", how="inner")
                common = base[["event_key",method]].merge(membership[["event_key","alternate_label"]], on="event_key", validate="one_to_one")
                assigned = common[method].ge(0) & common["alternate_label"].ge(0)
                resolution_rows.append({"method":method,"perturbation":kind,"value":value,"n_common_events":len(common),
                    "fraction_reference_events":len(common)/len(base) if len(base) else np.nan,
                    "ari_same_events_including_noise":adjusted_rand_score(common[method],common["alternate_label"]) if len(common)>1 else np.nan,
                    "ari_common_assigned":adjusted_rand_score(common.loc[assigned,method],common.loc[assigned,"alternate_label"]) if assigned.sum()>1 else np.nan,
                    "common_assigned_fraction":assigned.mean(), "n_grid_quarters":len(alt_train)})
    resolution = pd.DataFrame(resolution_rows)
    resolution.to_csv(root / "reports/stability/grid_and_magnitude_ari.csv", index=False)
    return params, resolution

<a id="s3"></a>
## 3. Jalankan eksperimen sensitivitas

ARI tinggi dapat didominasi kelompok besar/noise. Karena itu ARI seluruh data dan subset assigned dilaporkan bersama coverage. Tidak ada cutoff ARI universal untuk menyatakan pola stabil.

In [3]:
parameter_ari, spatial_ari = stability(ROOT)
display(parameter_ari)
display(spatial_ari)

,method,perturbation,value,n_same_rows,ari_all,ari_common_assigned,common_assigned_fraction
0,kmeans,n_clusters,5,13471,0.894107,0.894107,1.000000
1,kmeans,n_clusters,6,13471,1.000000,1.000000,1.000000
2,kmeans,n_clusters,7,13471,0.575450,0.575450,1.000000
3,kmeans,random_seed,7,13471,0.993341,0.993341,1.000000
4,kmeans,random_seed,21,13471,0.992369,0.992369,1.000000
5,kmeans,random_seed,42,13471,1.000000,1.000000,1.000000
6,kmeans,random_seed,84,13471,0.994747,0.994747,1.000000
7,kmeans,random_seed,123,13471,0.992369,0.992369,1.000000
8,dbscan,eps_and_min_samples,eps_x0.8;min_samples=10,13471,0.761684,0.770151,0.894737
9,dbscan,eps_and_min_samples,eps_x0.8;min_samples=20,13471,0.726206,0.783991,0.844629


,method,perturbation,value,n_common_events,fraction_reference_events,ari_same_events_including_noise,ari_common_assigned,common_assigned_fraction,n_grid_quarters
0,kmeans,grid_km,25.0,36146,1.000000,0.442776,0.442776,1.000000,21348
1,dbscan,grid_km,25.0,36146,1.000000,0.218412,0.114520,0.698307,21348
2,kmeans,grid_km,50.0,36146,1.000000,1.000000,1.000000,1.000000,13471
3,dbscan,grid_km,50.0,36146,1.000000,1.000000,1.000000,0.727328,13471
4,kmeans,grid_km,100.0,36146,1.000000,0.319266,0.319266,1.000000,6539
5,dbscan,grid_km,100.0,36146,1.000000,0.318332,0.181273,0.584684,6539
6,kmeans,min_magnitude,2.5,36146,1.000000,0.645576,0.645576,1.000000,15219
7,dbscan,min_magnitude,2.5,36146,1.000000,0.724377,0.811623,0.702318,15219
8,kmeans,min_magnitude,3.0,36146,1.000000,1.000000,1.000000,1.000000,13471
9,dbscan,min_magnitude,3.0,36146,1.000000,1.000000,1.000000,0.727328,13471


<a id="s4"></a>
## 4. Sensitivitas kebijakan pemilihan DBSCAN

Tabel ini mengulang seleksi dari kandidat training dengan beberapa batas ukuran cluster dan fragmentasi. Ini menunjukkan seberapa bergantung hasil pada kebijakan screening yang dipilih.

In [4]:
display(pd.read_csv(ROOT / "reports/stability/dbscan_selection_policy.csv"))

,max_clusters,min_cluster_size,status,eps,min_samples,n_clusters,noise_fraction,silhouette
0,10,5,candidate,0.505041,20.0,7.0,0.090416,0.129396
1,10,20,candidate,0.505041,20.0,7.0,0.090416,0.129396
2,10,50,no_candidate,NaN,NaN,NaN,NaN,NaN
3,20,5,candidate,0.505041,20.0,7.0,0.090416,0.129396
4,20,20,candidate,0.505041,20.0,7.0,0.090416,0.129396
5,20,50,no_candidate,NaN,NaN,NaN,NaN,NaN
6,50,5,candidate,0.172769,20.0,42.0,0.431816,0.366648
7,50,20,candidate,0.505041,20.0,7.0,0.090416,0.129396
8,50,50,no_candidate,NaN,NaN,NaN,NaN,NaN
9,300,5,candidate,0.104398,10.0,95.0,0.454755,0.497585


<a id="s5"></a>
## 5. Sensitivitas jumlah event per grid-triwulan

Profil dari satu atau dua event mudah mencerminkan diskretisasi fitur dan variasi sampling. Uji dukungan minimal 1/3/5/10 event: fit ulang scaler dan model hanya pada training yang lolos, dengan k/eps/min_samples tetap. ARI dibandingkan pada baris grid-triwulan yang sama dan coverage dilaporkan. Jika hanya sedikit baris lolos, klaim stabilitas tidak dapat digeneralisasi ke semua grid.

In [5]:
display(pd.read_csv(ROOT / "reports/stability/minimum_event_support.csv"))

,method,min_events_per_quarter,n_same_grid_quarters,fraction_reference_rows,n_clusters,noise_fraction,ari_same_grid_quarters,ari_common_assigned,common_assigned_fraction
0,kmeans,1,13471,1.000000,6,0.000000,1.000000,1.000000,1.000000
1,dbscan,1,13471,1.000000,7,0.090416,1.000000,1.000000,0.909584
2,kmeans,3,4042,0.300052,6,0.000000,0.330899,0.330899,1.000000
3,dbscan,3,4042,0.300052,2,0.381000,0.638837,1.000000,0.619000
4,kmeans,5,1809,0.134288,6,0.000000,0.067089,0.067089,1.000000
5,dbscan,5,1809,0.134288,2,0.533997,0.440148,0.000000,0.466003
6,kmeans,10,435,0.032292,6,0.000000,0.007853,0.007853,1.000000
7,dbscan,10,435,0.032292,1,0.800000,0.437773,1.000000,0.200000


## Referensi dan batas interpretasi

[Dokumentasi EMSC](https://www.seismicportal.eu/fdsn-wsevent.html), [clustering scikit-learn](https://scikit-learn.org/stable/modules/clustering.html), dan docs/metodologi.md. Hasil adalah profil aktivitas katalog dalam bounding box dan periode yang dipilih. Interpretasi risiko, prediksi kejadian gempa, serta klaim perubahan fisik memerlukan bukti tambahan.